# 08 - Targeted features: king-attack pressure, pins, game phase

Five new features in `extract_features_v3()` ([src/features.py](../src/features.py)), on top of the 34 from round 2:

- **`attackers_near_king_diff`** — unique *pieces* (not squares) attacking the king's 3x3 zone. Complements `king_safety_diff` (attacked squares — one queen can cover several) with actual attacking force.
- **`pins_diff`** — pieces pinned to their king, via `python-chess`'s built-in `board.is_pinned()`.
- **`total_non_pawn_material`** / **`is_endgame`** — a phase measure based on remaining material rather than move number (`fullmove_number` conflates a fast-trading game with a slow closed one).
- **`material_diff_x_endgame`** — explicit interaction term (`material_diff × is_endgame`), mainly for logistic regression's benefit — tree models can already discover this interaction via sequential splits, but a linear model can't.

Skipped for now (per our discussion): battery/alignment detection (rook-behind-queen) — real concept, but meaningfully more code for uncertain payoff.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.features import extract_features_v3, add_features_v3

train_df = pd.read_csv('../data/processed/train.csv', low_memory=False)
test_df = pd.read_csv('../data/processed/test.csv', low_memory=False)

train_df.shape, test_df.shape

((822936, 19), (208378, 19))

## Sanity check on pins and game phase

**Correction from the first version of this notebook**: the original sanity check assumed 1. e4 e5 2. Nf3 Nc6 3. Bb5 creates an absolute pin on the c6 knight. It doesn't — black's `d7` pawn is still on the same diagonal between the knight and the king, so the knight moving away wouldn't actually expose the king to check (the pawn already blocks that line). `pins_diff` correctly returned 0 there; the test case was wrong, not the code. Using a real pin instead: a lone rook and king vs. a knight and king, with the knight sitting directly between the rook and its own king on an open file.

Also verifying the `is_endgame` fix: it's now a **per-side** check (`PER_SIDE_ENDGAME_THRESHOLD = 13`, i.e. roughly queen+rook or queen+minor or less, on *both* sides) rather than the original combined-total threshold, which could misfire when one side kept nearly everything.

In [2]:
import chess

# a genuine pin: white rook e1 + king f1, black king e8 + knight e5 pinned on
# the open e-file, and a low-material endgame-type position to check is_endgame
pin_fen = '4k3/8/8/4n3/8/8/8/4RK2 b - - 0 1'

new_feature_names = [
    'attackers_near_king_diff', 'pins_diff',
    'total_non_pawn_material', 'is_endgame', 'material_diff_x_endgame',
]
feats = extract_features_v3(pin_fen)
{k: feats[k] for k in new_feature_names}

{'attackers_near_king_diff': 0,
 'pins_diff': 1,
 'total_non_pawn_material': 8,
 'is_endgame': 1,
 'material_diff_x_endgame': 2}

## Extract for train and test
Benchmarked at ~0.50ms/row (~8.7min combined) — the attacker/pin computations are the most expensive part of the whole feature set so far.

In [3]:
%%time
train_df = add_features_v3(train_df)
test_df = add_features_v3(test_df)

train_df.shape, test_df.shape

CPU times: user 8min 48s, sys: 5.77 s, total: 8min 54s
Wall time: 8min 55s


((822936, 58), (208378, 58))

## Check correlations and endgame prevalence

In [4]:
print('share of positions flagged as endgame:', train_df['is_endgame'].mean())
print()
print(train_df[new_feature_names + ['white_win']].corr()['white_win'].drop('white_win').sort_values(key=abs, ascending=False))

share of positions flagged as endgame: 0.21522937385167254

material_diff_x_endgame     0.254723
attackers_near_king_diff    0.194929
pins_diff                   0.018976
total_non_pawn_material     0.011852
is_endgame                 -0.007074
Name: white_win, dtype: float64


## Save enriched splits

In [5]:
train_df.to_csv('../data/processed/train_features_v3.csv', index=False)
test_df.to_csv('../data/processed/test_features_v3.csv', index=False)
print('saved.')

saved.
